# Step 1 — Pilot rollout 与生成长度验收

生成 `200 prompts × 2 rollouts = 400 trajectories`。本阶段只解决生成长度与截断问题，不训练。

**Go/No-Go：** finished rate ≥95%、撞到 2048-token 上限的比例 <5%、字段和样本数完整。

In [ ]:
# @title Step 01.1 — 初始化运行环境
from pathlib import Path
import gc
import importlib.util
import json
import os
import shutil
import subprocess
import sys

REPO = Path("/content/ZIP-RC-Colab")
ZIP_PY = Path("/content/mamba/envs/zip/bin/python")
REPO_URL = "https://github.com/wtree101/ZIP-RC-Colab.git"
REPO_BRANCH = "main"
SYNC_REPO = True

if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO)], check=True)
elif SYNC_REPO:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

if not ZIP_PY.exists():
    raise FileNotFoundError(
        f"未找到 {ZIP_PY}。请先建立 ZIP-RC 的 mamba 环境，再重新运行本 Notebook。"
    )

kernel_required = ["torch", "numpy", "pandas", "pyarrow", "matplotlib", "sklearn", "psutil"]
kernel_missing = [name for name in kernel_required if importlib.util.find_spec(name) is None]
if kernel_missing:
    raise ModuleNotFoundError(f"Colab kernel 缺少可视化依赖: {kernel_missing}")

env_check = subprocess.run(
    [
        str(ZIP_PY),
        "-c",
        (
            "import importlib.util, json; "
            "mods=['torch','vllm','transformers','datasets','pandas','pyarrow']; "
            "print(json.dumps([m for m in mods if importlib.util.find_spec(m) is None]))"
        ),
    ],
    check=True,
    capture_output=True,
    text=True,
)
env_missing = json.loads(env_check.stdout.strip())
if env_missing:
    raise ModuleNotFoundError(f"zip mamba 环境缺少依赖: {env_missing}")

os.environ["ZIPRC_PYTHON"] = str(ZIP_PY)

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import torch
from IPython.display import display

sys.path.insert(0, str(REPO / "notebooks"))
from ziprc_notebook_utils import (
    gate,
    gate_frame,
    load_config,
    model_artifacts_exist,
    read_jsonl,
    require_columns,
    rolling_edges,
    run_repo,
    save_stage_report,
)

print("Repository:", REPO)
print("ZIP Python:", ZIP_PY)

CONFIG = load_config(REPO)
print("Experiment:", CONFIG["experiment_name"])


In [ ]:
# @title Step 01.2 — 生成 pilot rollouts
RUN_STAGE = True
output_path = REPO / CONFIG["paths"]["pilot"]
if RUN_STAGE:
    run_repo(
        REPO,
        "python3", "src/generate_ziprc_rollouts.py",
        "--model", CONFIG["model_id"],
        "--dataset", CONFIG["dataset"],
        "--split", CONFIG["split"],
        "--prompt-column", CONFIG["prompt_column"],
        "--answer-column", CONFIG["answer_column"],
        "--out", output_path,
        "--max-num-prompts", CONFIG["pilot_prompts"],
        "--thinking-samples", 0,
        "--non-thinking-samples", CONFIG["pilot_rollouts_per_prompt"],
        "--temperature", CONFIG["temperature"],
        "--min-p", CONFIG["min_p"],
        "--max-model-len", CONFIG["generation_max_model_len"],
        "--max-new-tokens", CONFIG["max_output_tokens"],
        "--max-num-seqs", CONFIG["max_num_seqs"],
        "--dtype", CONFIG["dtype"],
        "--dp-size", 1, "--tp-size", 1,
    )
print("Artifact:", output_path)

In [ ]:
# @title Step 01.3 — 可视化生成长度与截断率
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

df = pd.read_parquet(output_path)
required = ["prompt_idx", "prompt", "answer", "response", "length", "finished", "reasoning_enabled", "input_ids", "label_positions"]
missing = require_columns(df, required)
if missing:
    raise ValueError(f"缺少列: {missing}")

expected_rows = int(CONFIG["pilot_prompts"]) * int(CONFIG["pilot_rollouts_per_prompt"])
finished_rate = float(df["finished"].mean())
cap_rate = float((df["length"] >= int(CONFIG["max_output_tokens"])).mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["length"], bins=30, color="#4c78a8", edgecolor="white")
axes[0].axvline(CONFIG["max_output_tokens"], color="#e45756", linestyle="--", label="output cap")
axes[0].set(title="Pilot response length", xlabel="output tokens", ylabel="rollouts")
axes[0].legend()
rates = pd.Series({"finished": finished_rate, "hit output cap": cap_rate})
rates.plot.bar(ax=axes[1], color=["#49beaa", "#ef767a"], ylim=(0, 1))
axes[1].axhline(0.95, color="#49beaa", linestyle=":")
axes[1].axhline(0.05, color="#ef767a", linestyle=":")
axes[1].set(title="Completion / truncation gates", ylabel="fraction")
plt.tight_layout()
plt.show()

display(df["length"].describe(percentiles=[.5, .9, .95, .99]).to_frame("tokens").round(2))
display(df[["prompt", "response", "length", "finished"]].head(5))

checks = [
    gate("字段完整", not missing, f"missing={missing}"),
    gate("样本数完整", len(df) == expected_rows, f"{len(df)}/{expected_rows}"),
    gate("Prompt 数完整", df["prompt_idx"].nunique() == CONFIG["pilot_prompts"], f"{df['prompt_idx'].nunique()} prompts"),
    gate("Finished rate ≥95%", finished_rate >= 0.95, f"{finished_rate:.1%}", kind="scientific"),
    gate("撞输出上限 <5%", cap_rate < 0.05, f"{cap_rate:.1%}", kind="scientific"),
]
display(gate_frame(checks))
save_stage_report(REPO, "01_pilot_generation", checks, {"rows": len(df), "finished_rate": finished_rate, "cap_rate": cap_rate, "p95_length": float(df['length'].quantile(.95))})